### CELL 1 input and ouput path defined

In [1]:
#CELL 1 input and ouput path defined
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR  = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### CELL 2 important imports

In [2]:
# CELL 2 imports
from __future__ import annotations

# Standard library imports
import glob
import json
import math
import os
import random
from collections import Counter
from copy import deepcopy
from itertools import combinations
from pathlib import Path
from typing import Any

# Third-party library imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py_stringmatching as sm
from dotenv import load_dotenv
from gensim.parsing.preprocessing import (
    lower_to_unicode, preprocess_string, strip_multiple_whitespaces,
    strip_numeric, strip_punctuation, strip_tags
)
from langchain_openai import ChatOpenAI
from openai import OpenAI
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Local/Project-specific imports
from PyDI.informationextraction import LLMExtractor
from PyDI.io import load_json
from PyDI.normalization import load_normalization_spec
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaTranslator

# Configuration
np.random.seed(42)
random.seed(42)
tqdm.pandas()

c:\Users\hussa\anaconda3\envs\hiwi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### CELL 3 loading the normalized dataset files 

In [3]:
#Cell3 loading the normalized dataset files (4 files total 3012 rows where 812 are Unique and Product 1 has everything that all other 3 files have)

products_1_cleaned = load_json(INPUT_DIR / "data_cleaned_final" / "dataset_1_normalized.json")
products_1_cleaned.attrs["dataset_name"] = "products_1"

products_2_cleaned = load_json(INPUT_DIR / "data_cleaned_final" / "dataset_2_normalized.json")
products_2_cleaned.attrs["dataset_name"] = "products_2"

products_3_cleaned = load_json(INPUT_DIR / "data_cleaned_final" / "dataset_3_normalized.json")
products_3_cleaned.attrs["dataset_name"] = "products_3"

products_4_cleaned = load_json(INPUT_DIR / "data_cleaned_final" / "dataset_4_normalized.json")
products_4_cleaned.attrs["dataset_name"] = "products_4"

local_norm_datasets = [products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned]

my_id_universe = set()
for df in local_norm_datasets:
    my_id_universe.update(df['id'].unique())

products_1_cleaned['id'] = products_1_cleaned['id'].astype('int64')
products_2_cleaned['id'] = products_2_cleaned['id'].astype('int64')
products_3_cleaned['id'] = products_3_cleaned['id'].astype('int64')
products_4_cleaned['id'] = products_4_cleaned['id'].astype('int64')

print(f"Total unique IDs in my dataset: {len(my_id_universe)}")

products_1_cleaned['source_file'] = 'products_1_cleaned.csv'
products_2_cleaned['source_file'] = 'products_2_cleaned.csv'
products_3_cleaned['source_file'] = 'products_3_cleaned.csv'
products_4_cleaned['source_file'] = 'products_4_cleaned.csv'

#quick check to see if all cluster_ids are already in products_1_cleaned 
all_cluster_ids = set()
for df in local_norm_datasets:
    all_cluster_ids.update(df['cluster_id'].unique())

missing_ids = all_cluster_ids - set(products_1_cleaned['cluster_id'])
if missing_ids:
    print(f"Missing cluster_ids in products_1_cleaned: {len(missing_ids)}")
else:
    print("All cluster_ids are present in products_1_cleaned")

Total unique IDs in my dataset: 3012
All cluster_ids are present in products_1_cleaned


### CELL 4 loading the seen and unseen csv files from DBSCAN using hard negatives

In [4]:
# CELL 4 loading the seen and unseen csv files from DBSCAN using hard negatives

# Load your actual results
seen_mapping = pd.read_csv(OUTPUT_DIR / 'dbscan_results' / 'seen_dbscan_mapping.csv')
unseen_mapping = pd.read_csv(OUTPUT_DIR / 'dbscan_results' / 'unseen_dbscan_mapping.csv')

# These are your 93 entities
unseen = set(unseen_mapping['cluster_id']) 

# Combine them for the hard negative lookup
all_mappings = pd.concat([seen_mapping, unseen_mapping])
dbscan_dict = all_mappings.set_index('cluster_id')['dbscan_cluster'].to_dict()


# Use the results from your DBSCAN experiment
unseen_entities = set(unseen_mapping['cluster_id']) # Your 93 entities
seen_entities = set(seen_mapping['cluster_id'])     # Your 410 entities

# The 'active' universe of products we are actually using
active_cluster_ids = unseen_entities | seen_entities

### CELL 5 PREPROCESSING and SIMILARITY 

In [5]:
# CELL 5 PREPROCESSING and SIMILARITY 

# ============================================================
# PREPROCESSING + SIMILARITY
# also using py_stringmatching dependency for the first 3 metrics, and OpenAI embeddings for the 4th metric instead of FastText
# ============================================================


CUSTOM_FILTERS = [
    lambda x: x.lower(),
    strip_tags,
    strip_punctuation,
    strip_multiple_whitespaces
]

class OpenAIEmbeddingSim:
    def n_similarity(self, tokens_a, tokens_b):
        return openai_small_sim(tokens_a, tokens_b)

OPENAI_MODEL = OpenAIEmbeddingSim()    

def preprocess(title):
    """Matches WDC repo's gensim preprocessing pipeline exactly."""
    return preprocess_string(lower_to_unicode(str(title)), CUSTOM_FILTERS)


load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Cache to save money and time
_embedding_cache = {}

def get_embedding_small(text):
    """Fetches text-embedding-3-small vectors (1536 dimensions)."""
    if not text or not text.strip():
        return np.zeros(1536) 
    
    key = text.strip().lower()

    if key not in _embedding_cache:
        response = client.embeddings.create(
            input=[text],
            model="text-embedding-3-small"
        )
        _embedding_cache[key] = response.data[0].embedding

    return np.array(_embedding_cache[key])

def openai_small_sim(tokens_a, tokens_b):
    """
    Semantic similarity using text-embedding-3-small.
    Replaces the FastText/SoftTFIDF slot in the WDC rotation.
    """
    text_a = " ".join(tokens_a)
    text_b = " ".join(tokens_b)
    
    if not text_a or not text_b:
        return 0.0
    
    vec_a = get_embedding_small(text_a)
    vec_b = get_embedding_small(text_b)
    
    # Cosine Similarity
    dot = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    
    return float(dot / (norm_a * norm_b)) if norm_a > 0 and norm_b > 0 else 0.0

# Define the WDC-style 4-metric rotation
CC_SIMILARITIES = [
    sm.Cosine(),
    sm.GeneralizedJaccard(threshold=0.7),
    sm.Dice(),
    OPENAI_MODEL
] 


### CELL 6 merge dataset and preprocess titles

In [6]:
# CELL 6 merge dataset and preprocess titles
# ============================================================
# SETUP — merge datasets + preprocess titles
# ============================================================

# 1. Merge all files
all_data = pd.concat(local_norm_datasets, ignore_index=True)
#caught a mistake here where I was dropping duplicates by 'id' as the same product can appear in multiple files with the same id but different cluster_id. 

all_data['dbscan_cluster'] = all_data['cluster_id'].map(dbscan_dict)


# 2. Create initial global_id
all_data['global_id'] = all_data['source_file'] + "_" + all_data['id'].astype(str)

# 3. Handle duplicates WITHIN the same source (if any) 
# THIS MUST HAPPEN BEFORE DICTIONARY CREATION
if all_data['global_id'].duplicated().any():
    print("Warning: Duplicate global_ids found! Resetting index to ensure uniqueness.")
    # Appending index to make them truly unique
    all_data['global_id'] = all_data['global_id'] + "_" + all_data.index.astype(str)

# 4. Validation: Ensure one ID doesn't point to two different clusters
# (Essential for ground truth integrity in Professor Bizer's projects)
cluster_check = all_data.groupby('global_id')['cluster_id'].nunique()
if (cluster_check > 1).any():
    print(f"CRITICAL: {len(cluster_check[cluster_check > 1])} IDs are assigned to multiple clusters!")

# 5. Preprocess titles
all_data['title_processed'] = all_data['title'].apply(preprocess)

# 6. Build lookup dictionaries (Now using the finalized global_id)
id_to_cluster = all_data.set_index('global_id')['cluster_id'].to_dict()
id_to_title = all_data.set_index('global_id')['title_processed'].to_dict()


print(f"Total Records: {len(all_data)}")
print(f"Unique Global IDs: {all_data['global_id'].nunique()}")
print(f"Total Clusters: {all_data['cluster_id'].nunique()}")

Total Records: 3012
Unique Global IDs: 3012
Total Clusters: 812


In [7]:
# CELL 7: STRATIFIED SPLIT ensuring no cluster_id leakage across Train/Val/Test, and that all 'Unseen' entities are strictly in Test


# 1. Load the mappings
all_mappings = pd.concat([
    pd.read_csv(OUTPUT_DIR / 'dbscan_results' / 'seen_dbscan_mapping.csv'), 
    pd.read_csv(OUTPUT_DIR / 'dbscan_results' / 'unseen_dbscan_mapping.csv')
])
dbscan_map = all_mappings.set_index('cluster_id')['dbscan_cluster'].to_dict()
neighborhood_counts = all_mappings['dbscan_cluster'].value_counts()

# 2. Categorize Clusters from seen_entities
# A cluster is 'social' if it shares a neighborhood with others
social_seen = [c for c in seen_entities if c in dbscan_map and neighborhood_counts[dbscan_map[c]] > 1]
# A cluster is 'isolated' if it has no neighbors or wasn't in the mapping
isolated_seen = [c for c in seen_entities if c not in social_seen]

print(f"Stratification: {len(social_seen)} 'Social' clusters vs {len(isolated_seen)} 'Isolated' clusters.")

# --- Helper Function for Safe Splitting ---
def safe_split(cluster_list, train_ratio=0.5, random_state=42):
    if not cluster_list:
        return set(), set(), set()
    
    # If list is too small to split into 3 parts, put all in train
    if len(cluster_list) < 3:
        return set(cluster_list), set(), set()
    
    # Split 50% Train, 50% Remainder nstead of 70 30 as we needed mroe test data 
    train, rem = train_test_split(cluster_list, train_size=train_ratio, random_state=random_state)
    
    # Split Remainder 50/50 into Val and Test
    val, test = train_test_split(rem, train_size=0.50, random_state=random_state)
    
    return set(train), set(val), set(test)

# 3. Perform the splits
s_train, s_val, s_test = safe_split(social_seen)
i_train, i_val, i_test = safe_split(isolated_seen)

# 4. Combine sets and add the reserved 'Unseen' entities strictly to Test
train_c = s_train.union(i_train)
val_c   = s_val.union(i_val)
test_c  = s_test.union(i_test).union(set(unseen_entities))

print("-" * 30)
print(f"Final Cluster Counts:")
print(f"  Train: {len(train_c)} clusters ({len(s_train)} social)")
print(f"  Val:   {len(val_c)} clusters ({len(s_val)} social)")
print(f"  Test:  {len(test_c)} clusters ({len(s_test)} social + {len(unseen_entities)} unseen)")

# 5. Verification Check
overlap = train_c.intersection(test_c)
if not overlap:
    print("✅ LEAKAGE CHECK: PASSED. Clusters are strictly isolated.")
else:
    print(f"⚠️ WARNING: {len(overlap)} clusters leaked!")

Stratification: 416 'Social' clusters vs 0 'Isolated' clusters.
------------------------------
Final Cluster Counts:
  Train: 208 clusters (208 social)
  Val:   104 clusters (104 social)
  Test:  174 clusters (104 social + 70 unseen)
✅ LEAKAGE CHECK: PASSED. Clusters are strictly isolated.


In [8]:
# ============================================================
# Cell 8: SIMILARITY SETUP

# SIMILARITY SETUP (replaces random.sample within neighborhood)
# Cosine, GenJaccard, Dice + OpenAI embeddings as 4th metric
# ============================================================
# CC_SIMILARITIES = [ #kept here again for ref, but already defined above
#     sm.Cosine(),
#     sm.GeneralizedJaccard(threshold=0.7),
#     sm.Dice(),
#     OPENAI_MODEL
# ] 

def apply_sim(sim_obj, tokens_a, tokens_b):
    if not tokens_a or not tokens_b:
        return 0.0
    if hasattr(sim_obj, 'get_sim_score'):
        return sim_obj.get_sim_score(tokens_a, tokens_b)
    elif hasattr(sim_obj, 'n_similarity'):  
        return sim_obj.n_similarity(tokens_a, tokens_b)
    return 0.0
            
def get_split_corpus(split_df, full_corpus):
    """
    In WDC logic, if split_df is None (as in your cell), 
    we treat full_corpus as the pre-filtered split.
    """
    if split_df is None:
        return full_corpus.copy()
    
    # If split_df is a list of IDs, filter the corpus
    return full_corpus[full_corpus['cluster_id'].isin(split_df)].copy()

def is_hard_negative(cid_left, cid_right, dbscan_dict):
    """
    Matches the WDC 'Corner Case' logic:
    True if two products are in the same DBSCAN group but different Clusters.
    """
    if cid_left == cid_right:
        return False # They are the same product (Positive)
    
    # Check if they share the same 'neighborhood' (DBSCAN global_id)
    group_left = dbscan_dict.get(cid_left)
    group_right = dbscan_dict.get(cid_right)
    
    return (group_left is not None) and (group_left == group_right)



def get_openai_embedding(text, client, model="text-embedding-3-small"):
    """Fetch OpenAI embedding for a single text string."""
    text = str(text).replace("\n", " ")
    response = client.embeddings.create(input=[text], model=model)
    return np.array(response.data[0].embedding)

def cosine_sim_vectors(vec_a, vec_b):
    """Cosine similarity between two numpy vectors."""
    denom = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    return float(np.dot(vec_a, vec_b) / denom) if denom else 0.0

def build_embedding_cache(corpus_df, client, model="text-embedding-3-small"):
    """
    Pre-compute OpenAI embeddings for all titles in the corpus.
    Done once per split to avoid repeated API calls.
    """
    print("  Pre-computing OpenAI embeddings...")
    cache = {}
    for _, row in tqdm(corpus_df.iterrows(), total=len(corpus_df), desc="Embeddings"):
        pid = row['global_id']
        if pid not in cache:
            cache[pid] = get_openai_embedding(row['title'], client, model)
    return cache


def score_candidates(anchor_tokens, anchor_id, candidate_rows,
                     embedding_cache, n_metrics=4):
    """
    Replicates repo's metric rotation:
    Scores every candidate with all 4 metrics and returns
    one sorted list per metric (descending similarity).

    Metrics:
      [0] Cosine (token frequency)
      [1] GeneralizedJaccard
      [2] Dice
      [3] OpenAI embedding cosine
    """
    scores_per_metric = [[] for _ in range(n_metrics)]

    anchor_emb = embedding_cache.get(anchor_id)

    for idx, cand_row in candidate_rows.iterrows():
        tok_b = cand_row['title_processed']
        cid   = idx  # index is 'global_id'

        # Use the apply_sim dispatcher instead of calling the objects directly
        scores_per_metric[0].append((idx, apply_sim(CC_SIMILARITIES[0], anchor_tokens, tok_b)))
        scores_per_metric[1].append((idx, apply_sim(CC_SIMILARITIES[1], anchor_tokens, tok_b)))
        scores_per_metric[2].append((idx, apply_sim(CC_SIMILARITIES[2], anchor_tokens, tok_b)))

        if anchor_emb is not None and cid in embedding_cache:
            emb_score = cosine_sim_vectors(anchor_emb, embedding_cache[cid])
        else:
            emb_score = 0.0
        scores_per_metric[3].append((idx, emb_score))

    # Sort each metric's list descending — repo picks from top of sorted list
    sorted_per_metric = [
        [item[0] for item in sorted(lst, key=lambda x: x[1], reverse=True)]
        for lst in scores_per_metric
    ]
    return sorted_per_metric


def pick_hard_negatives_with_rotation(anchor_id, anchor_tokens,
                                       hard_candidates_df, sub_corpus_wo,
                                       embedding_cache, all_neg,
                                       neg_limit, n_metrics=4):
    """
    Replicates repo's while loop + random metric rotation:
    — Randomly pick a metric
    — Walk down that metric's ranked list until an unseen,
      different-cluster candidate is found
    — Repeat until neg_limit hard negatives collected

    Returns list of chosen (anchor_id, neg_id) pairs.
    """
    if hard_candidates_df.empty:
        return []

    sorted_per_metric = score_candidates(
        anchor_tokens, anchor_id, hard_candidates_df, embedding_cache, n_metrics)

    selected_negs          = []
    selected_neg_set       = set()
    already_selected_clus  = set()
    max_attempts           = neg_limit * n_metrics * 10  # safety cap
    attempts               = 0

    while len(selected_negs) < neg_limit and attempts < max_attempts:
        attempts += 1
        # Repo: random.sample(range(4), 1)[0]
        metric_idx = random.sample(range(n_metrics), 1)[0]

        for candidate_idx in sorted_per_metric[metric_idx]:
            if candidate_idx not in hard_candidates_df.index:
                continue
            cand_row = hard_candidates_df.loc[candidate_idx]
            rel_clu  = cand_row['cluster_id']

            if (rel_clu not in already_selected_clus
                    and (anchor_id, candidate_idx) not in all_neg
                    and (candidate_idx, anchor_id) not in all_neg
                    and (anchor_id, candidate_idx) not in selected_neg_set
                    and (candidate_idx, anchor_id) not in selected_neg_set):

                chosen = random.sample(
                    [(anchor_id, candidate_idx), (candidate_idx, anchor_id)], 1)[0]
                selected_negs.append(chosen)
                selected_neg_set.add(chosen)
                already_selected_clus.add(rel_clu)
                break  # found one for this metric attempt — move to next iteration

    return selected_negs


# ============================================================
# UPDATED build_and_materialise_star
# Fixes point 2 (similarity ranking) and point 3 (metric rotation)
# ============================================================

def build_and_materialise_star(split_df, full_corpus, split_name,
                                embedding_cache, neg_limit=4):
    """
    Star Schema pair generation with WDC-style hard negative mining:
    — Hub (products_1_cleaned) paired with Spokes (files 2,3,4)
    — Hard negatives ranked by 4 metrics with random rotation (fixes pts 2+3)
    — 70% hard, 30% random negatives per hub offer
    """
    corpus_df = get_split_corpus(split_df, full_corpus)
    corpus_df['title_processed'] = corpus_df['title'].apply(preprocess)
    corpus_indexed = corpus_df.set_index('global_id', drop=False)    

    all_pairs = []
    all_neg   = set()   # global dedup set — matches repo's (i, selected) checks
    cids      = corpus_df['cluster_id'].unique()

    num_hard = int(neg_limit * 0.7)  # = 2 when neg_limit=4
    num_rnd  = neg_limit - num_hard  # = 2 when neg_limit=4

    for cid in tqdm(cids, desc=f"Building {split_name} (Star Schema)"):
        sub_corpus = corpus_indexed[corpus_indexed['cluster_id'] == cid]
        if sub_corpus.empty:
            continue
            
        other_clusters = corpus_indexed[corpus_indexed['cluster_id'] != cid]
        
        # 1. Grab neighborhood from the audited column
        db_group = sub_corpus['dbscan_cluster'].iloc[0]

        hub_rows   = sub_corpus[sub_corpus['source_file'] == 'products_1_cleaned.csv']
        spoke_rows = sub_corpus[sub_corpus['source_file'] != 'products_1_cleaned.csv']

        # STEP 1: Star positives
        for h_id in hub_rows['global_id']:
            for s_id in spoke_rows['global_id']:
                pair = random.sample([(h_id, s_id), (s_id, h_id)], 1)[0]
                all_pairs.append({'id1': pair[0], 'id2': pair[1], 'label': 1})

        # STEP 2: Star negatives
        if hub_rows.empty or other_clusters.empty:
            continue

        spoke_other = other_clusters[other_clusters['source_file'] != 'products_1_cleaned.csv']

        for h_id in hub_rows['global_id']:
            anchor_tokens = hub_rows.loc[h_id]['title_processed']

            # A) HARD negatives — only if they share a neighborhood
            hard_chosen = []
            if pd.notna(db_group):
                # Filter only candidates in the SAME DBSCAN neighborhood
                hard_candidates_df = spoke_other[spoke_other['dbscan_cluster'] == db_group]
                
                hard_chosen = pick_hard_negatives_with_rotation(
                    anchor_id = h_id,
                    anchor_tokens = anchor_tokens,
                    hard_candidates_df = hard_candidates_df,
                    sub_corpus_wo = spoke_other,
                    embedding_cache = embedding_cache,
                    all_neg = all_neg,
                    neg_limit = num_hard,
                    n_metrics = 4
                )

            for pair in hard_chosen:
                all_neg.add(pair)
                all_pairs.append({'id1': pair[0], 'id2': pair[1], 'label': 0})

            # B) RANDOM negatives — any spoke outside this cluster
            #    Exclude already-selected hard neg IDs
            hard_ids = {p[1] if p[0] == h_id else p[0] for p in hard_chosen}
            rnd_pool = spoke_other[~spoke_other.index.isin(hard_ids)]

            rnd_chosen_count = 0
            rnd_attempts     = 0
            while rnd_chosen_count < num_rnd and rnd_attempts < 50:
                rnd_attempts += 1
                if rnd_pool.empty:
                    break
                rnd_id = rnd_pool.sample(1)['global_id'].iloc[0]
                if ((h_id, rnd_id) not in all_neg
                        and (rnd_id, h_id) not in all_neg):
                    pair = random.sample([(h_id, rnd_id), (rnd_id, h_id)], 1)[0]
                    all_neg.add(pair)
                    all_pairs.append({'id1': pair[0], 'id2': pair[1], 'label': 0})
                    rnd_chosen_count += 1

    # Materialise
    final_df = pd.DataFrame(all_pairs)
    if final_df.empty:
        return pd.DataFrame(columns=['id1', 'id2', 'label'])

    left_data  = corpus_indexed.loc[final_df['id1']].reset_index(drop=True)
    right_data = corpus_indexed.loc[final_df['id2']].reset_index(drop=True)

    output = left_data.join(right_data, lsuffix='_left', rsuffix='_right')
    output['label'] = final_df['label'].values
    output['is_hard_negative'] = output.apply(
        lambda r: r['label'] == 0 and is_hard_negative(
            r['cluster_id_left'], r['cluster_id_right'], dbscan_dict), axis=1)

    return output.sample(frac=1, random_state=42).reset_index(drop=True)


# ============================================================
# EXECUTION — build embedding cache once per split, then generate
# ============================================================
client = OpenAI()  # uses OPENAI_API_KEY env variable


# Create split-specific corpora (The 'Active' data for each split)
train_df_corpus = all_data[all_data['cluster_id'].isin(train_c)]
val_df_corpus   = all_data[all_data['cluster_id'].isin(val_c)]
test_df_corpus  = all_data[all_data['cluster_id'].isin(test_c)]

print(f"Records: Train={len(train_df_corpus)}, Val={len(val_df_corpus)}, Test={len(test_df_corpus)}")


print("Building embedding cache for each split...")
train_emb_cache = build_embedding_cache(train_df_corpus, client)
val_emb_cache   = build_embedding_cache(val_df_corpus,   client)
test_emb_cache  = build_embedding_cache(test_df_corpus,  client)

print("Generating pairs...")
train_pairs = build_and_materialise_star(None, train_df_corpus, 'train', train_emb_cache)
val_pairs   = build_and_materialise_star(None, val_df_corpus,   'val',   val_emb_cache)
test_pairs  = build_and_materialise_star(None, test_df_corpus,  'test',  test_emb_cache)

final_train, final_val, final_test = train_pairs, val_pairs, test_pairs

print(f"\nTrain: {len(train_pairs)} (hards: {train_pairs['is_hard_negative'].sum()})")
print(f"Val:   {len(val_pairs)}   (hards: {val_pairs['is_hard_negative'].sum()})")
print(f"Test:  {len(test_pairs)}  (hards: {test_pairs['is_hard_negative'].sum()})")


# After build_and_materialise_star runs
for split_name, split_df in [('train', final_train), ('val', final_val), ('test', final_test)]:
    total_neg = (split_df['label'] == 0).sum()
    hard_neg  = split_df['is_hard_negative'].sum()
    print(f"{split_name}: {hard_neg}/{total_neg} negatives are hard ({hard_neg/total_neg:.1%})")


Records: Train=832, Val=416, Test=611
Building embedding cache for each split...
  Pre-computing OpenAI embeddings...


Embeddings: 100%|██████████| 832/832 [01:55<00:00,  7.21it/s]


  Pre-computing OpenAI embeddings...


Embeddings: 100%|██████████| 416/416 [00:54<00:00,  7.63it/s]


  Pre-computing OpenAI embeddings...


Embeddings: 100%|██████████| 611/611 [01:25<00:00,  7.17it/s]


Generating pairs...


Building test (Star Schema): 100%|██████████| 174/174 [00:02<00:00, 63.55it/s]



Train: 1366 (hards: 340)
Val:   650   (hards: 143)
Test:  1051  (hards: 290)
train: 340/742 negatives are hard (45.8%)
val: 143/338 negatives are hard (42.3%)
test: 290/614 negatives are hard (47.2%)


In [13]:
# ============================================================
# # CELL 9 (FINAL): EXPORT, NORMALIZATION & INTEGRITY CHECKS
# ============================================================


# 1. Setup Directories
# Ensure OUTPUT_DIR is defined (e.g., OUTPUT_DIR = Path("./output"))
base_export_path = OUTPUT_DIR / "entity_matching_final_ground_truth"
per_pair_path = base_export_path / "per_file_splits"
base_export_path.mkdir(parents=True, exist_ok=True)
per_pair_path.mkdir(parents=True, exist_ok=True)

FILE_MAP = {
    "products_1_cleaned.csv": "prod1",
    "products_2_cleaned.csv": "prod2",
    "products_3_cleaned.csv": "prod3",
    "products_4_cleaned.csv": "prod4",
}

def process_and_export(final_df, split_name):
    if final_df is None or final_df.empty:
        print(f"Skipping {split_name} - DataFrame is empty.")
        return
    
    # --- A) Save Master Split (JSONL for ML & CSV for Human Inspection) ---
    master_export = final_df.rename(columns={'id_left': 'id1', 'id_right': 'id2'})
    
    # JSONL Compressed
    json_file = base_export_path / f"{split_name}_pairs.json.gz"
    master_export.to_json(json_file, lines=True, orient='records', compression='gzip')
    
    # CSV (Restored from Cell 14)
    csv_file = base_export_path / f"{split_name}_gt.csv"
    master_export.to_csv(csv_file, index=False)
    
    print(f"✅ Saved Master {split_name}: {len(final_df)} pairs")

    # --- B) Generate Normalized Per-File Splits (Vectorized) ---
    df = final_df.copy()
    df["src_left"]  = df["source_file_left"].map(FILE_MAP)
    df["src_right"] = df["source_file_right"].map(FILE_MAP)
    
    for right_tag in ["prod2", "prod3", "prod4"]:
        mask = ((df["src_left"] == "prod1") & (df["src_right"] == right_tag)) | \
               ((df["src_left"] == right_tag) & (df["src_right"] == "prod1"))
        
        subset = df[mask].copy()
        if subset.empty: continue
        
        # VECTORIZED NORMALIZATION: Force prod1 to the left (id1)
        # Identify rows where prod1 is on the right side
        swap_mask = subset['src_right'] == 'prod1'
        
        # Initialize id1/id2 with defaults
        subset['id1'] = subset['id_left']
        subset['id2'] = subset['id_right']
        
        # For rows where prod1 is on the right, flip the IDs
        subset.loc[swap_mask, 'id1'] = subset.loc[swap_mask, 'id_right']
        subset.loc[swap_mask, 'id2'] = subset.loc[swap_mask, 'id_left']
        
        # Clean up columns for export
        out_df = subset[['id1', 'id2', 'label']].copy()
        if 'is_hard_negative' in subset.columns:
            out_df['is_hard'] = subset['is_hard_negative']
        
        fname = per_pair_path / f"prod1_to_{right_tag}_{split_name}.csv"
        out_df.to_csv(fname, index=False)
        
        hard_count = out_df['is_hard'].sum() if 'is_hard' in out_df.columns else 0
        print(f"    ∟ {fname.name}: {len(out_df)} pairs (Hards: {hard_count})")

# 2. Run Export
process_and_export(final_train, "train")
process_and_export(final_val,   "val")
process_and_export(final_test,  "test")

# 3. Final Research Integrity Audit (Your Improved Logic)
print("\n" + "="*50)
print("FINAL RESEARCH INTEGRITY AUDIT")
print("="*50)

# Check 1: Comprehensive Cluster Leakage (Train vs Test)
train_clusters = set(final_train['cluster_id_left']).union(set(final_train['cluster_id_right']))
test_clusters  = set(final_test['cluster_id_left']).union(set(final_test['cluster_id_right']))
leakage = train_clusters.intersection(test_clusters)

# Check 2: Star Schema (Hub presence)
def is_star_schema(df):
    if df is None or df.empty: return True
    return all((df['source_file_left'] == 'products_1_cleaned.csv') | 
               (df['source_file_right'] == 'products_1_cleaned.csv'))

star_ok = is_star_schema(final_train) and is_star_schema(final_test)

# Report results
if not leakage:
    print("💎 LEAKAGE CHECK: PASSED (Clusters are 100% isolated)")
else:
    print(f"⚠️ LEAKAGE CHECK: FAILED ({len(leakage)} clusters overlap between Train and Test!)")

if star_ok:
    print("⭐ STAR SCHEMA: PASSED (All pairs contain Hub 'products_1')")
else:
    print("⚠️ STAR SCHEMA: FAILED (Some pairs are Spoke-to-Spoke)")

print("="*50)

✅ Saved Master train: 1366 pairs
    ∟ prod1_to_prod2_train.csv: 509 pairs (Hards: 166)
    ∟ prod1_to_prod3_train.csv: 445 pairs (Hards: 104)
    ∟ prod1_to_prod4_train.csv: 412 pairs (Hards: 70)
✅ Saved Master val: 650 pairs
    ∟ prod1_to_prod2_val.csv: 248 pairs (Hards: 86)
    ∟ prod1_to_prod3_val.csv: 190 pairs (Hards: 27)
    ∟ prod1_to_prod4_val.csv: 212 pairs (Hards: 30)
✅ Saved Master test: 1051 pairs
    ∟ prod1_to_prod2_test.csv: 463 pairs (Hards: 155)
    ∟ prod1_to_prod3_test.csv: 379 pairs (Hards: 98)
    ∟ prod1_to_prod4_test.csv: 209 pairs (Hards: 37)

FINAL RESEARCH INTEGRITY AUDIT
💎 LEAKAGE CHECK: PASSED (Clusters are 100% isolated)
⭐ STAR SCHEMA: PASSED (All pairs contain Hub 'products_1')


In [14]:
# cell 10 checking the generated GT files for sanity (counts, hard negative ratio, etc.)
def analyze_gt_file(path, df_left_source, df_right_source):
    # Load the Ground Truth
    df = pd.read_csv(path)
    
    # Ensure IDs from the CSV are strings to match 'global_id'
    df['id1'] = df['id1'].astype(str)
    df['id2'] = df['id2'].astype(str)
    
    # ... (keep your mapping logic if you want to see titles) ...
    # 1. Map the IDs back to their Titles
    # Bring in title_left
    df = df.merge(df_left_source[['global_id', 'title']], left_on='id1', right_on='global_id', how='left')
    df = df.rename(columns={'title': 'title_left'}).drop(columns=['global_id'])    
    # Bring in title_right
    df = df.merge(df_right_source[['global_id', 'title']], left_on='id2', right_on='global_id', how='left')
    df = df.rename(columns={'title': 'title_right'}).drop(columns=['global_id'])
    
    # 2. Basic Counts
    counts = df['label'].value_counts()
    positives = counts.get(1, 0)
    negatives = counts.get(0, 0)
    total = len(df)
    
    # 3. USE THE SAVED HARD NEGATIVE FLAG 
    # This matches exactly what your generation script produced
    if 'is_hard' in df.columns:
        hard_negatives_count = df['is_hard'].sum()
    else:
        # Fallback if column name is different
        hard_negatives_count = df.get('is_hard_negative', pd.Series([0]*len(df))).sum()
    
    print("-" * 50)
    print(f"File: {os.path.basename(path)}")
    print("-" * 50)
    print(f"Total Pairs:      {total}")
    print(f"Positive Matches:  {positives:<4} ({positives/total:>5.1%})")
    print(f"Negative Matches:  {negatives:<4} ({negatives/total:>5.1%})")
    print(f"  -> True Hard Negs: {hard_negatives_count:<4} ({hard_negatives_count/max(1, negatives):>5.1%} of negs)")

# Example usage:
# train
analyze_gt_file(per_pair_path / "prod1_to_prod2_train.csv", train_df_corpus, train_df_corpus)
analyze_gt_file(per_pair_path / "prod1_to_prod3_train.csv", train_df_corpus, train_df_corpus)
analyze_gt_file(per_pair_path / "prod1_to_prod4_train.csv", train_df_corpus, train_df_corpus)

# test
analyze_gt_file(per_pair_path / "prod1_to_prod2_test.csv", test_df_corpus, test_df_corpus)
analyze_gt_file(per_pair_path / "prod1_to_prod3_test.csv", test_df_corpus, test_df_corpus)
analyze_gt_file(per_pair_path / "prod1_to_prod4_test.csv", test_df_corpus, test_df_corpus)

# val
analyze_gt_file(per_pair_path / "prod1_to_prod2_val.csv", val_df_corpus, val_df_corpus)
analyze_gt_file(per_pair_path / "prod1_to_prod3_val.csv", val_df_corpus, val_df_corpus)
analyze_gt_file(per_pair_path / "prod1_to_prod4_val.csv", val_df_corpus, val_df_corpus)


--------------------------------------------------
File: prod1_to_prod2_train.csv
--------------------------------------------------
Total Pairs:      509
Positive Matches:  208  (40.9%)
Negative Matches:  301  (59.1%)
  -> True Hard Negs: 166  (55.1% of negs)
--------------------------------------------------
File: prod1_to_prod3_train.csv
--------------------------------------------------
Total Pairs:      445
Positive Matches:  208  (46.7%)
Negative Matches:  237  (53.3%)
  -> True Hard Negs: 104  (43.9% of negs)
--------------------------------------------------
File: prod1_to_prod4_train.csv
--------------------------------------------------
Total Pairs:      412
Positive Matches:  208  (50.5%)
Negative Matches:  204  (49.5%)
  -> True Hard Negs: 70   (34.3% of negs)
--------------------------------------------------
File: prod1_to_prod2_test.csv
--------------------------------------------------
Total Pairs:      463
Positive Matches:  174  (37.6%)
Negative Matches:  289  (62.4%)

In [15]:
# ============================================================
# Cell 11: GT difficulty analysis using Generalized Jaccard
# ============================================================

import glob
import os
from sklearn.metrics import f1_score

gj = sm.GeneralizedJaccard(threshold=0.7)

def get_gj_sim(tokens_a, tokens_b):
    if not tokens_a or not tokens_b:
        return 0.0
    return gj.get_sim_score(tokens_a, tokens_b)

def analyse_test_difficulty(test_pairs_df, all_data, label=""):
    # Index all_data by global_id for the primary title lookup
    corpus = all_data.set_index('global_id')
    results = test_pairs_df.copy()
    
    id1_col = 'id_left'
    id2_col = 'id_right'
    hard_col = 'is_hard_negative'

    # Ensure IDs are strings for the corpus .loc lookup
    results[id1_col] = results[id1_col].astype(str)
    results[id2_col] = results[id2_col].astype(str)

    results['title_sim'] = results.apply(
        lambda r: get_gj_sim(
            corpus.loc[r[id1_col]]['title_processed'] if r[id1_col] in corpus.index else [],
            corpus.loc[r[id2_col]]['title_processed'] if r[id2_col] in corpus.index else []
        ), axis=1
    )

    pos      = results[results['label'] == 1]
    neg      = results[results['label'] == 0]
    hard_neg = results[results[hard_col] == True] if hard_col in results.columns else pd.DataFrame()

    print("=" * 60)
    print(f"DIFFICULTY REPORT: {label}")
    print("=" * 60)

    print(f"\n[NEGATIVES] Total: {len(neg)} | Hard Mined: {len(hard_neg)}")
    if not hard_neg.empty:
        print(f"   Avg GJ Sim (Hard Negs): {hard_neg['title_sim'].mean():.3f}")
        print(f"   Genuinely Hard (>0.5 GJ): {(hard_neg['title_sim'] > 0.5).sum()} pairs")
        print(f"   Moderate (0.3-0.5 GJ):    {((hard_neg['title_sim'] > 0.3) & (hard_neg['title_sim'] <= 0.5)).sum()} pairs")
        print(f"   Easy (<0.3 GJ):            {(hard_neg['title_sim'] < 0.3).sum()} pairs")

    print(f"\n[POSITIVES] Total: {len(pos)}")
    if not pos.empty:
        print(f"   Avg GJ Sim (Positives): {pos['title_sim'].mean():.3f}")
        print(f"   Low-Overlap (<0.4 GJ):  {(pos['title_sim'] < 0.4).sum()} pairs")
        print(f"   High-Overlap (>0.7 GJ): {(pos['title_sim'] > 0.7).sum()} pairs")

    preds = (results['title_sim'] >= 0.5).astype(int)
    f1 = f1_score(results['label'], preds)
    print(f"\n[BASELINE] GJ matcher (0.5 threshold) F1: {f1:.3f}")

    if not pos.empty and not hard_neg.empty:
        margin = pos['title_sim'].mean() - hard_neg['title_sim'].mean()
        print(f"\n[HARDNESS MARGIN] {margin:.3f}")
        
    print(f"\n[TOP 3 HARDEST NEGATIVES]")
    if not hard_neg.empty:
        for _, row in hard_neg.nlargest(3, 'title_sim').iterrows():
            t_l = corpus.loc[row[id1_col]]['title'][:60] if row[id1_col] in corpus.index else 'N/A'
            t_r = corpus.loc[row[id2_col]]['title'][:60] if row[id2_col] in corpus.index else 'N/A'
            print(f"   sim={row['title_sim']:.3f} | {t_l}  vs  {t_r}")

    return results


# ── EXECUTION ──────────────────────────────────────────────

# 1. Prepare meta_lookup using RAW numeric IDs to ensure it matches the CSV content
meta_lookup = pd.concat([final_train, final_val, final_test])[
    ['id_left', 'id_right', 'is_hard_negative']
].drop_duplicates().copy()

# This is the "Magic" line: Strip the prefix from the memory IDs so they match the CSV 'id1'/'id2'
meta_lookup['id_left_raw'] = meta_lookup['id_left'].astype(str).str.split('_').str[-1]
meta_lookup['id_right_raw'] = meta_lookup['id_right'].astype(str).str.split('_').str[-1]

all_files = sorted(glob.glob(str(per_pair_path / "prod1_to_*.csv")))

for file_path in all_files:
    fname = os.path.basename(file_path)
    # Detect the correct prefix for the right-side product
    spoke_tag = fname.split('_to_')[1].split('_')[0] 
    spoke_prefix = f"products_{spoke_tag[-1]}_cleaned.csv"
    hub_prefix = "products_1_cleaned.csv"
    
    print("\n\n" + "#"*80)
    print(f"CONNECTION: {fname}")
    print("#"*80)

    # Load CSV (which contains raw numbers like 84634220)
    df = pd.read_csv(file_path).rename(columns={'id1': 'id_left_raw', 'id2': 'id_right_raw'})
    df['id_left_raw'] = df['id_left_raw'].astype(str)
    df['id_right_raw'] = df['id_right_raw'].astype(str)
    
    # 2. Merge with meta_lookup using the raw numbers to get the 'is_hard_negative' flag
    report_df = df.merge(meta_lookup, on=['id_left_raw', 'id_right_raw'], how='left')

    # 3. RECONSTRUCT GLOBAL IDs specifically for the title lookup inside the function
    
    report_df['id_left'] = hub_prefix + "_" + report_df['id_left_raw']
    report_df['id_right'] = spoke_prefix + "_" + report_df['id_right_raw']

    analyse_test_difficulty(report_df, all_data, label=fname)



################################################################################
CONNECTION: prod1_to_prod2_test.csv
################################################################################
DIFFICULTY REPORT: prod1_to_prod2_test.csv

[NEGATIVES] Total: 289 | Hard Mined: 67
   Avg GJ Sim (Hard Negs): 0.469
   Genuinely Hard (>0.5 GJ): 22 pairs
   Moderate (0.3-0.5 GJ):    36 pairs
   Easy (<0.3 GJ):            9 pairs

[POSITIVES] Total: 174
   Avg GJ Sim (Positives): 0.505
   Low-Overlap (<0.4 GJ):  50 pairs
   High-Overlap (>0.7 GJ): 23 pairs

[BASELINE] GJ matcher (0.5 threshold) F1: 0.475

[HARDNESS MARGIN] 0.036

[TOP 3 HARDEST NEGATIVES]
   sim=0.895 | Kingston A2000 500GB SSD 3D NAND M.2 2280 PCIe NVMe 3.0 x4 I  vs  Kingston A2000 1TBB SSD 3D NAND M.2 2280 PCIe NVMe 3.0 x4 In
   sim=0.750 | HDD extern WD My Passport, 4TB, 2,5" USB 3.0, portocaliu  vs  WD 4TB My Passport 2.5" USB 3.0 Portable Hard Drive - White
   sim=0.714 | Kingston KC2000 M.2 NVMe - 500GB  vs  Kingsto

### Checking how hard the test set is one last time after adjustemnts

In [16]:


# 1. SETUP PATHS
BASE_INPUT = INPUT_DIR / "data"
BASE_OUTPUT = OUTPUT_DIR /'entity_matching_final_ground_truth' / 'per_file_splits' 

# Initialize Similarity Measure
gj = sm.GeneralizedJaccard(threshold=0.7)

def load_json_as_dict(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    # Create a lookup: {id: {title: "...", cluster_id: "..."}}
    return {str(item['id']): item for item in data}

def manual_audit(gt_filename, prod1_json, prod2_json, top_n=10):
    print(f"\n{'='*90}\nAUDITING: {gt_filename}\n{'='*90}")
    
    # Load Data
    gt_df = pd.read_csv(BASE_OUTPUT / gt_filename)
    p1_lookup = load_json_as_dict(BASE_INPUT / prod1_json)
    p2_lookup = load_json_as_dict(BASE_INPUT / prod2_json)
    
    results = []
    
    for _, row in gt_df.iterrows():
        id1, id2, label = str(row['id1']), str(row['id2']), row['label']
        
        # Get raw data
        item1 = p1_lookup.get(id1, {})
        item2 = p2_lookup.get(id2, {})
        
        t1 = item1.get('title', 'N/A')
        t2 = item2.get('title', 'N/A')
        
        # Calculate similarity (tokenizing by space)
        sim = gj.get_sim_score(t1.lower().split(), t2.lower().split())
        
        results.append({
            'id1': id1, 'id2': id2, 'label': label,
            'title1': t1, 'title2': t2, 'sim': sim
        })
    
    audit_df = pd.DataFrame(results)
    
    # --- DISPLAY LOGIC ---
    
    # 1. HARD NEGATIVES (Label 0, but high Similarity)
    print(f"\n🔥 TOP {top_n} HARD NEGATIVES (Trap cases: should look like matches)")
    hard_negs = audit_df[audit_df['label'] == 0].nlargest(top_n, 'sim')
    for _, r in hard_negs.iterrows():
        print(f"[{r['sim']:.3f}] ID1:{r['id1']} vs ID2:{r['id2']}")
        print(f"   T1: {r['title1'][:80]}")
        print(f"   T2: {r['title2'][:80]}\n")

    # 2. HARD POSITIVES (Label 1, but low Similarity)
    print(f"\n💎 TOP {top_n} HARD POSITIVES (Diverse cases: very different titles)")
    hard_pos = audit_df[audit_df['label'] == 1].nsmallest(top_n, 'sim')
    for _, r in hard_pos.iterrows():
        print(f"[{r['sim']:.3f}] ID1:{r['id1']} vs ID2:{r['id2']}")
        print(f"   T1: {r['title1'][:80]}")
        print(f"   T2: {r['title2'][:80]}\n")



# Define the audit list (GT file, Prod1 JSON, Spoke JSON)
test_files = [
    ("prod1_to_prod2_test.csv", "products_1.json", "products_2.json"),
    ("prod1_to_prod3_test.csv", "products_1.json", "products_3.json"),
    ("prod1_to_prod4_test.csv", "products_1.json", "products_4.json")
]

for gt_file, p1_js, p2_js in test_files:
    # This calls the manual_audit function we wrote in the previous step
    manual_audit(
        gt_filename=gt_file, 
        prod1_json=p1_js, 
        prod2_json=p2_js,
        top_n=5 # Showing top 5 for brevity
    )



AUDITING: prod1_to_prod2_test.csv

🔥 TOP 5 HARD NEGATIVES (Trap cases: should look like matches)
[0.957] ID1:59897811 vs ID2:56179024
   T1: ASUS GeForce GTX 1660 6GB Phoenix Boost Graphics Card
   T2: ASUS GeForce GTX 1650 4GB Phoenix Boost Graphics Card

[0.882] ID1:2468281 vs ID2:61774550
   T1: Kingston A2000 500GB SSD 3D NAND M.2 2280 PCIe NVMe 3.0 x4 Internal Solid State 
   T2: Kingston A2000 1TBB SSD 3D NAND M.2 2280 PCIe NVMe 3.0 x4 Internal Solid State D

[0.858] ID1:58208007 vs ID2:10041015
   T1: MSI RADEON RX 5700 XT MECH OC 8GB GDDR6
   T2: MSI Radeon RX 5500 XT Mech 8G OC

[0.785] ID1:54649100 vs ID2:56179024
   T1: Asus GeForce GTX 1650 OC Phoenix 4GB GPU/Graphics Card
   T2: ASUS GeForce GTX 1650 4GB Phoenix Boost Graphics Card

[0.747] ID1:6557275 vs ID2:31617338
   T1: M3 Portable USB 3.0 Hard Drive - 4TB
   T2: Maxtor M3 Portable USB 3.0 Hard Drive, 1TB


💎 TOP 5 HARD POSITIVES (Diverse cases: very different titles)
[0.058] ID1:30624986 vs ID2:4241237
   T1: EVGA N

## what we can do fusion on

In [86]:
from itertools import combinations
import pandas as pd
from tqdm import tqdm
from collections import Counter
import re 
# 1. SETUP MEASURES
gj = sm.GeneralizedJaccard(threshold=0.8)
me = sm.MongeElkan()
set_tok = sm.WhitespaceTokenizer(return_set=True)
list_tok = sm.WhitespaceTokenizer(return_set=False)

def is_numeric_conflict(val1, val2, tolerance=0.2):
    try:
        v1, v2 = float(val1), float(val2)
        if v1 == 0 or v2 == 0: return v1 != v2
        return (abs(v1 - v2) / max(v1, v2)) > tolerance
    except: return False

def is_text_conflict(val1, val2, measure, threshold=0.8):
    s1, s2 = str(val1).strip().lower(), str(val2).strip().lower()
     # Normalize spaces around brackets/parentheses
    s1 = re.sub(r'\s*([(\[])\s*', r'\1', s1)
    s2 = re.sub(r'\s*([(\[])\s*', r'\1', s2)
    # Remove parentheses/brackets entirely (e.g. 'M.2 (2280)' -> 'M.2 2280')
    s1 = re.sub(r'[(){}\[\]]', ' ', s1).split()
    s2 = re.sub(r'[(){}\[\]]', ' ', s2).split()
    s1 = ' '.join(s1)
    s2 = ' '.join(s2)
    if s1 == s2 or not s1 or not s2: return False
    # Select tokenizer based on measure type
    t1, t2 = (list_tok.tokenize(s1), list_tok.tokenize(s2)) if isinstance(measure, sm.MongeElkan) \
             else (set_tok.tokenize(s1), set_tok.tokenize(s2))
    if not t1 or not t2: return False
    try:
        score = measure.get_sim_score(t1, t2) if hasattr(measure, 'get_sim_score') else measure.get_raw_score(t1, t2)
        return score < threshold
    except: return False

# 2. STAR-SCHEMA PAIR GENERATION
all_star_positives = []
hub_file = 'products_1_cleaned.csv'

for cid, group in all_data.groupby('cluster_id'):
    if len(group) > 1:
        hub_record = group[group['source_file'] == hub_file]
        spokes = group[group['source_file'] != hub_file]
        
        if not hub_record.empty:
            h_gid = hub_record.iloc[0]['global_id']
            for _, spoke in spokes.iterrows():
                all_star_positives.append({
                    'id_left': h_gid,
                    'id_right': spoke['global_id'],
                    'label': 1
                })

all_positives_df = pd.DataFrame(all_star_positives)
print(f"Total Star-Schema pairs generated: {len(all_positives_df)}")

# 3. CORE MINING ENGINE
def mine_fusion_data(pairs_df, full_records_df, target_types=['GPU', 'HDD', 'SSD', 'USB_STICK']):
    identity_cols = ['brand', 'model', 'model_number', 'chipset_name','bus_type', 'interface_type', 'form_factor']
    spec_cols = ['vram_gb', 'storage_gb', 'read_speed_mb_s', 'write_speed_mb_s']

    # CRITICAL: O(1) lookup using the unique global_id
    records = full_records_df.set_index('global_id')
    fusion_candidates = []
    consistency_tracker = Counter()

    example_limit = 10
    examples_printed = 0

    for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="Mining Conflicts"):
        lid, rid = row['id_left'], row['id_right']

        if lid not in records.index or rid not in records.index:
            continue

        l = records.loc[lid] # The Hub (Prod 1)
        r = records.loc[rid] # The Spoke (Prod 2, 3, or 4)
        
        p_type = l['product_type']
        if p_type not in target_types: continue

        conflict_list = []
        null_fill_list = []

        for col in (identity_cols + spec_cols):
            lv, rv = l.get(col), r.get(col)
            
            # CHECK A: Value vs Null (Consistency)
            if pd.notna(lv) and pd.isna(rv):
                null_fill_list.append(f"{col}_from_Hub")
                consistency_tracker[l['source_file']] += 1 # Credit Prod 1
            elif pd.isna(lv) and pd.notna(rv):
                null_fill_list.append(f"{col}_from_Spoke")
                consistency_tracker[r['source_file']] += 1 # Credit the Spoke
            
            # CHECK B: Value vs Value (Conflict)
            elif pd.notna(lv) and pd.notna(rv):
                # Using Monge-Elkan for structured identity fields
                if col in identity_cols and is_text_conflict(lv, rv, me, 0.85):
                    conflict_list.append(col)
                # Using Numeric logic for specs
                elif col in spec_cols and is_numeric_conflict(lv, rv, 0.2):
                    conflict_list.append(col)
        
        # Final Title Similarity Check (Jaccard)
   
        #wont use title for conflicts as almost all are diff and very noisy. so will just have it here for now as info only.
        title_conflict = is_text_conflict(l['title'], r['title'], gj, 0.8)    
        # --- ADD THIS INSIDE THE LOOP IN Section 3 ---
        if conflict_list and examples_printed < example_limit:
            print(f"\n🚩 EXAMPLE CONFLICT {examples_printed + 1}: Cluster {l['cluster_id']}")
            print(f"   Match: {l['source_file']} <--> {r['source_file']}")
            
            for col in conflict_list:
                lv, rv = l.get(col), r.get(col)
                if col in identity_cols:
                    t1, t2 = (list_tok.tokenize(str(lv).lower()), list_tok.tokenize(str(rv).lower()))
                    score = me.get_raw_score(t1, t2)
                    print(f"   ↳ [Identity Fail] {col}: '{lv}' vs '{rv}' (Sim: {score:.2f})")
                elif col in spec_cols:
                    v1, v2 = float(lv), float(rv)
                    diff = abs(v1 - v2) / max(v1, v2)
                    print(f"   ↳ [Spec Mismatch] {col}: {lv} vs {rv} (Var: {diff:.1%})")
            
            examples_printed += 1
            if examples_printed == example_limit:
                print("\n... [will print only the first 10 examples to avoid cluter] ...")

        if conflict_list or null_fill_list:
            fusion_candidates.append({
                'id_left': lid, 'id_right': rid,
                'source_left': l['source_file'],  
                'source_right': r['source_file'],
                'cluster_id': l['cluster_id'],
                'product_type': p_type,
                'conflicts': ", ".join(conflict_list),
                'null_fills': ", ".join(null_fill_list),
                'num_issues': len(conflict_list) + len(null_fill_list),
                'title_conflict': title_conflict,        # info only, doesn't affect ranking
                'title_left': l['title'], 'title_right': r['title'],
                'description_left': l.get('description'), 
                'description_right': r.get('description')
            })

    results_df = pd.DataFrame(fusion_candidates)
    
    if not results_df.empty:
    # --- STRATIFIED SELECTION LOGIC ---
        cluster_ranking = results_df.groupby(['product_type', 'cluster_id'])['num_issues'].max().reset_index()
        
        final_top_ids = []
        sampling_map = {}

        for p_type in target_types:
            type_subset = cluster_ranking[cluster_ranking['product_type'] == p_type].sort_values('num_issues', ascending=False)
            
            # 1. Super Hard (Top 35 - 70%)
            super_hard = type_subset.head(35)
            for cid in super_hard['cluster_id']:
                sampling_map[cid] = "Super Hard (Top 70%)"
            
            # 2. Medium/Random (Next 15 - 30%)
            remaining = type_subset.iloc[35:]
            if not remaining.empty:
                medium_random = remaining.sample(n=min(15, len(remaining)), random_state=42)
                for cid in medium_random['cluster_id']:
                    sampling_map[cid] = "Medium/Random (Bottom 30%)"
                final_top_ids.append(medium_random['cluster_id'])
            
            final_top_ids.append(super_hard['cluster_id'])

        top_cluster_ids = pd.concat(final_top_ids)
        final_selection = results_df[results_df['cluster_id'].isin(top_cluster_ids)].copy()
        
        # Mapping the difficulty tag back to the main dataframe
        final_selection['sampling_type'] = final_selection['cluster_id'].map(sampling_map)
        final_selection = final_selection.sort_values(['product_type', 'cluster_id', 'num_issues'], ascending=[True, True, False])

        print("\nSampling Distribution with 70/30 hard/random)")
        sample_stats = final_selection.groupby(['product_type', 'sampling_type'])['cluster_id'].nunique().unstack(fill_value=0)
        print(sample_stats)
        print("-"*30)
            
    else:
        final_selection = results_df

    consistency_summary = pd.DataFrame.from_dict(
        consistency_tracker, orient='index', columns=['contribution_score']
    ).sort_values('contribution_score', ascending=False)

    return final_selection, consistency_summary

# 4. RUN
fusion_final, consistency_report = mine_fusion_data(all_positives_df, all_data)

# TEMPORARY STATS CHECK 
temp_stats = fusion_final.groupby('cluster_id').apply(
    lambda x: set(x['source_left']) | set(x['source_right']),
    include_groups=False
)
universal_count = sum(1 for s in temp_stats if len(s) == 4)
print(f"📊 PROOF: {universal_count} out of 200 clusters actually have data from all 4 sources!")


# --- NEW AGGREGATE SUMMARY BLOCK ---
print("\n" + "="*40)
print("📊 FUSION DATA AGGREGATE REPORT")
print("="*40)

# 1. Total Counts
print(f"Total Clusters Mined: {fusion_final['cluster_id'].nunique()}")
print(f"Total Pairwise Comparisons: {len(fusion_final)}")

# 2. Conflict Type Frequency
all_conflicts = []
for c in fusion_final['conflicts'].dropna():
    all_conflicts.extend([x.strip() for x in c.split(',') if x.strip()])
conflict_counts = Counter(all_conflicts)

print("\nTop Conflict Attributes (Hard Mismatches):")
for attr, count in conflict_counts.most_common():
    print(f"  - {attr:15}: {count} pairs")

# 3. Null-Fill Frequency (Complementation)
all_fills = []
for f in fusion_final['null_fills'].dropna():
    all_fills.extend([x.strip() for x in f.split(',') if x.strip()])
fill_counts = Counter(all_fills)

print("\nTop Complementation Opportunities (Filling Holes):")
for attr, count in fill_counts.most_common(5):
    print(f"  - {attr:25}: {count} instances")

# 4. Source Reliability
print("\nSource Consistency Report (Contribution Score):")
print(consistency_report)
print("="*40)

# --- CLEAN UP IDs BEFORE EXPORT ---
def strip_global_prefix(val):
    if pd.isna(val): return val
    # This splits by the last underscore to handle "file_name_123"
    return str(val).split('_')[-1]

fusion_final['id_left'] = fusion_final['id_left'].apply(strip_global_prefix)
fusion_final['id_right'] = fusion_final['id_right'].apply(strip_global_prefix)


# Save files
fusion_final.to_csv(OUTPUT_DIR / 'fusion_data' / 'fusion_dataset_candidates4_after_dr_ralph_updt.csv', index=False)
consistency_report.to_csv(OUTPUT_DIR / 'fusion_data' / 'source_consistency_report4_after_dr_ralph_updt.csv')

Total Star-Schema pairs generated: 2200


Mining Conflicts:   7%|▋         | 164/2200 [00:00<00:01, 1632.40it/s]


🚩 EXAMPLE CONFLICT 1: Cluster 491
   Match: products_1_cleaned.csv <--> products_2_cleaned.csv
   ↳ [Identity Fail] interface_type: 'NVMe 1.3' vs 'NVMe' (Sim: 0.50)

🚩 EXAMPLE CONFLICT 2: Cluster 491
   Match: products_1_cleaned.csv <--> products_3_cleaned.csv
   ↳ [Identity Fail] interface_type: 'NVMe 1.3' vs 'NVMe' (Sim: 0.50)

🚩 EXAMPLE CONFLICT 3: Cluster 491
   Match: products_1_cleaned.csv <--> products_4_cleaned.csv
   ↳ [Identity Fail] model: 'XPG SX6000 Pro' vs 'SX6000 Pro' (Sim: 0.83)
   ↳ [Identity Fail] interface_type: 'NVMe 1.3' vs 'NVMe' (Sim: 0.50)

🚩 EXAMPLE CONFLICT 4: Cluster 3668
   Match: products_1_cleaned.csv <--> products_2_cleaned.csv
   ↳ [Identity Fail] form_factor: 'M.2 2280' vs '2280' (Sim: 0.76)

🚩 EXAMPLE CONFLICT 5: Cluster 3668
   Match: products_1_cleaned.csv <--> products_4_cleaned.csv
   ↳ [Identity Fail] form_factor: 'M.2 2280' vs 'M.2' (Sim: 0.76)

🚩 EXAMPLE CONFLICT 6: Cluster 5306
   Match: products_1_cleaned.csv <--> products_2_cleaned.csv
   ↳ 

Mining Conflicts: 100%|██████████| 2200/2200 [00:01<00:00, 1599.53it/s]


Sampling Distribution with 70/30 hard/random)
sampling_type  Medium/Random (Bottom 30%)  Super Hard (Top 70%)
product_type                                                   
GPU                                    15                    35
HDD                                    15                    35
SSD                                    15                    35
USB_STICK                              15                    35
------------------------------
📊 PROOF: 125 out of 200 clusters actually have data from all 4 sources!

📊 FUSION DATA AGGREGATE REPORT
Total Clusters Mined: 200
Total Pairwise Comparisons: 505

Top Conflict Attributes (Hard Mismatches):
  - model          : 215 pairs
  - form_factor    : 112 pairs
  - bus_type       : 84 pairs
  - interface_type : 53 pairs
  - brand          : 30 pairs
  - model_number   : 17 pairs
  - storage_gb     : 10 pairs
  - write_speed_mb_s: 7 pairs
  - read_speed_mb_s: 2 pairs
  - chipset_name   : 1 pairs

Top Complementation Opportuniti

In [87]:
import pandas as pd

# -----------------------------
# 1. LOAD YOUR GT FILE
# -----------------------------
df = pd.read_csv(OUTPUT_DIR / 'fusion_data' / 'fusion_dataset_candidates4_after_dr_ralph_updt.csv')

base_cols = ['id_left', 'id_right', 'source_left', 'source_right', 'cluster_id', 'product_type','num_issues','sampling_type']
new_df = df[base_cols].copy()

# -----------------------------
# 2. ADD EMPTY COLUMNS
# -----------------------------
new_columns = [
    'id','brand','title','description','price','priceCurrency','cluster_id',
    'url','title_description','model','model_number','product_type',
    'chipset_name','vram_gb','storage_gb','read_speed_mb_s','write_speed_mb_s',
    'bus_type','interface_type','width_mm','length_mm','height_mm','weight_g',
    'storage_connection_type','memory_type','color','form_factor',
    'gt_source_url','gt_source_2','filled', 'sampling_type' #filled is to know if i filled it before when comparing
]

for col in new_columns:
    if col not in new_df.columns:
        new_df[col] = ""

# -----------------------------
# 3. SHORTEN SOURCE NAMES
# -----------------------------
import re

def shorten_source(val):
    match = re.search(r'products_(\d+)_cleaned', str(val))
    return f"p{match.group(1)}" if match else val

new_df['source_left'] = new_df['source_left'].apply(shorten_source)
new_df['source_right'] = new_df['source_right'].apply(shorten_source)

# -----------------------------
# 4. BUILD LOOKUP TABLES
# -----------------------------
lookup_tables = {}

for df_local in local_norm_datasets:
    name = df_local.attrs["dataset_name"]  # products_1_cleaned
    short = f"p{name.split('_')[1]}"       # p1, p2, ...
    lookup_tables[short] = df_local.set_index("id")

# -----------------------------
# 5. HELPER: VALUE SELECTION
# -----------------------------
def choose_value(val1, val2):
    if pd.isna(val1) or val1 == "":
        return val2
    if pd.isna(val2) or val2 == "":
        return val1
    return val1 if len(str(val1)) >= len(str(val2)) else val2

# -----------------------------
# 6. OPTIONAL: CLUSTER-LEVEL FUSION 🔥
# -----------------------------
def get_cluster_records(cluster_id):
    results = []
    for df_local in lookup_tables.values():
        matches = df_local[df_local['cluster_id'] == cluster_id]
        if not matches.empty:
            results.append(matches)
    return pd.concat(results) if results else None

def best_from_cluster(cluster_df, col):
    if cluster_df is None or col not in cluster_df:
        return None
    
    vals = cluster_df[col].dropna()
    vals = vals[vals != ""]
    
    if vals.empty:
        return None
    
    # choose longest string
    return vals.loc[vals.astype(str).str.len().idxmax()]

# 7. ENRICHMENT FUNCTION (adjusted)
# -----------------------------
def enrich_row(row):
    try:
        # 1. Grab the specific pair records
        left = lookup_tables[row['source_left']].loc[row['id_left']]
        right = lookup_tables[row['source_right']].loc[row['id_right']]
        
        # 2. Grab every record in this cluster from all 4 files
        cluster_df = get_cluster_records(row['cluster_id'])
        
        # 3. Helper: Global search (Best from Cluster) -> Pairwise search (Best of 2)
        def get_best(col):
            # Try Cluster first (Global), then Pairwise (Local)
            return best_from_cluster(cluster_df, col) or choose_value(left.get(col), right.get(col))

        # 4. Return the "Golden" data for your GT file
        return pd.Series({
            'title': get_best('title'),
            'description': get_best('description'),
            'brand': get_best('brand'),
            'model_number': get_best('model_number'),
            'url': get_best('url'),
            'bus_type': get_best('bus_type'),
            'form_factor': get_best('form_factor')
        })
        
    except Exception as e:
        # Return Nones if the ID isn't found
        return pd.Series({col: None for col in ['title',  'description', 'brand', 'model_number', 'url', 'bus_type', 'form_factor']})


# -----------------------------
# 9. OPTIONAL DEBUG COLUMNS (VERY USEFUL)
# -----------------------------
def add_debug(row):
    try:
        left = lookup_tables[row['source_left']].loc[row['id_left']]
        right = lookup_tables[row['source_right']].loc[row['id_right']]
        
        return pd.Series({
            'title_left_raw': left.get('title'),
            'title_right_raw': right.get('title'),
            'desc_left_raw': left.get('description'), # NEW
            'desc_right_raw': right.get('description') # NEW
        })
    except:
        return pd.Series()
# -----------------------------
# 9. sort + deduplication logic to pick the "Messiest" row per cluster
# -----------------------------
fields = new_df.apply(enrich_row, axis=1)

cols_to_fill = ['title', 'description', 'brand', 'model_number', 'url', 'bus_type', 'form_factor']
new_df[cols_to_fill] = fields

# 1. Sort by num_issues (Descending) 
# This puts the comparison with the most conflicts/holes at the top of each cluster
new_df = new_df.sort_values(['cluster_id', 'num_issues'], ascending=[True, False])

# 2. Now drop duplicates
# Because we sorted, .drop_duplicates() will automatically keep the "Messiest" row
new_df = new_df.drop_duplicates(subset=['cluster_id']).copy()

# 3. Select your columns for the final labeling sheet
cols_to_keep = [
    'cluster_id', 'product_type', 'id_left', 'source_left',
    'brand', 'title','description', 'model_number', 'bus_type', 'form_factor', 'url', 'num_issues','filled', 'sampling_type',
]

# Add any other columns you want in your final labeling sheet
gt_final = new_df[cols_to_keep].copy()


debug_cols = new_df.apply(add_debug, axis=1)
new_df = pd.concat([new_df, debug_cols], axis=1)

# -----------------------------
# 10. SAVE FINAL FILE
# -----------------------------
new_df.to_csv(OUTPUT_DIR / 'fusion_data' / 'fusion_dataset_gt_half_fill4_after_dr_ralph_updt.csv', index=False)

print("✅ Ground truth file created with auto-filled values!")

✅ Ground truth file created with auto-filled values!


In [88]:
import pandas as pd
import re

# -----------------------------
# 1. LOAD & PREP
# -----------------------------
df_cand = pd.read_csv(OUTPUT_DIR / 'fusion_data' / 'fusion_dataset_gt_half_fill4_after_dr_ralph_updt.csv')
base_cols = ['id_left', 'id_right', 'source_left', 'source_right', 'cluster_id', 'product_type', 'num_issues', 'sampling_type']
new_df = df_cand[base_cols].copy()

# Shorten names
def shorten_source(val):
    match = re.search(r'products_(\d+)_cleaned', str(val))
    return f"p{match.group(1)}" if match else val

new_df['source_left'] = new_df['source_left'].apply(shorten_source)
new_df['source_right'] = new_df['source_right'].apply(shorten_source)

# -----------------------------
# 2. LOOKUPS & FUSION HELPERS
# -----------------------------
lookup_tables = {f"p{d.attrs['dataset_name'].split('_')[1]}": d.set_index("id") for d in local_norm_datasets}

def choose_value(val1, val2):
    if pd.isna(val1) or val1 == "": return val2
    if pd.isna(val2) or val2 == "": return val1
    return val1 if len(str(val1)) >= len(str(val2)) else val2

def get_cluster_records(cluster_id):
    results = [df_l[df_l['cluster_id'] == cluster_id] for df_l in lookup_tables.values()]
    results = [r for r in results if not r.empty]
    return pd.concat(results) if results else None

def best_from_cluster(cluster_df, col):
    if cluster_df is None or col not in cluster_df: return None
    vals = cluster_df[col].dropna()
    vals = vals[vals != ""]
    return vals.loc[vals.astype(str).str.len().idxmax()] if not vals.empty else None

# -----------------------------
# 3. ENRICHMENT & DEBUG FUNCTIONS
# -----------------------------
def enrich_row(row):
    try:
        left = lookup_tables[row['source_left']].loc[row['id_left']]
        cluster_df = get_cluster_records(row['cluster_id'])
        
        def get_best(col):
            # We use cluster_df (all sources) to find the absolute best technical string
            return best_from_cluster(cluster_df, col) or left.get(col)

        return pd.Series({
            'brand': get_best('brand'),
            'title': get_best('title'),
            'description': get_best('description'),
            'title_description': get_best('title_description'),
            'model': get_best('model'),
            'model_number': get_best('model_number'),
            'chipset_name': get_best('chipset_name'),
            'vram_gb': get_best('vram_gb'),
            'storage_gb': get_best('storage_gb'),
            'read_speed_mb_s': get_best('read_speed_mb_s'),
            'write_speed_mb_s': get_best('write_speed_mb_s'),
            'bus_type': get_best('bus_type'),
            'interface_type': get_best('interface_type'),
            'width_mm': get_best('width_mm'),
            'length_mm': get_best('length_mm'),
            'height_mm': get_best('height_mm'),
            'weight_g': get_best('weight_g'),
            'form_factor': get_best('form_factor'),
            'storage_connection_type': get_best('storage_connection_type'),
            'memory_type': get_best('memory_type'),
            'color': get_best('color'),
            'url': get_best('url'),
            'price': get_best('price'),
            'priceCurrency': get_best('priceCurrency'),
            'gt_source_url': "", # will fill manually
            'gt_source_2': "" # will fill manually
        })
    except:
        return pd.Series({col: "" for col in ['brand', 'title', 'description', 'title_description', 'model_number', 'url', 'bus_type', 'form_factor']})

def add_debug(row):
    try:
        l = lookup_tables[row['source_left']].loc[row['id_left']]
        r = lookup_tables[row['source_right']].loc[row['id_right']]
        return pd.Series({
            'title_left_raw': l.get('title'),
            'title_right_raw': r.get('title'),
            'desc_left_raw': l.get('description'),
            'desc_right_raw': r.get('description')
        })
    except:
        # Match the keys here for the error state
        return pd.Series({col: "" for col in [
            'brand', 'title', 'description', 'title_description', 'model', 'model_number', 'chipset_name',
            'vram_gb', 'storage_gb', 'read_speed_mb_s', 'write_speed_mb_s', 
            'bus_type', 'interface_type', 'width_mm', 'length_mm', 'height_mm', 
            'weight_g', 'form_factor', 'storage_connection_type', 'memory_type', 
            'color', 'url', 'price', 'priceCurrency', 'gt_source_url', 'gt_source_2','sampling_type'
        ]})
# -----------------------------
# 4. EXECUTE DEDUPLICATION (200 ROWS)
# -----------------------------


# First, sort so the "Messiest" pair for each cluster is on top
new_df = new_df.sort_values(['cluster_id', 'num_issues'], ascending=[True, False])

# Deduplicate to get exactly 1 row per product
gt_200 = new_df.drop_duplicates(subset=['cluster_id']).copy()

# Add a 'filled' column for your manual tracking
gt_200['filled'] = ""

# Apply Enrichment to the 200 rows
enriched = gt_200.apply(enrich_row, axis=1)
gt_200[enriched.columns] = enriched

# Apply Debug Columns to the 200 rows
debug = gt_200.apply(add_debug, axis=1)
gt_200[debug.columns] = debug

# -----------------------------
# 5. SAVE ONE FINAL AUDIT FILE
# -----------------------------
final_cols = [
      
    'id_left', 'id_right','source_left', 'source_right','cluster_id',
    'product_type', 'filled', #num_issues not needed #filled to check
    'brand', 'title', 'description', 'price','priceCurrency','url', 'title_description', 
    'model','model_number','chipset_name','vram_gb','storage_gb',
    'read_speed_mb_s','write_speed_mb_s', 'bus_type', 'interface_type', #done
    'length_mm','height_mm','width_mm','weight_g','storage_connection_type',
    'memory_type','color', 'form_factor', 'gt_source_url','gt_source_2', 'title_left_raw', 'title_right_raw', 
    'desc_left_raw', 'desc_right_raw','sampling_type'
]

#sort the gt by product type and cluster_id for easier navigation during labeling
gt_200 = gt_200.sort_values(['product_type', 'cluster_id'])



gt_200[final_cols].to_csv(OUTPUT_DIR / 'fusion_data' / 'fusion_dataset_gt_half_fill4_after_dr_ralph_updt.csv', index=False)

print(f"✅ Created file with {len(gt_200)} unique products")

✅ Created file with 200 unique products
